# Resume Role Predictor-LLaMA 3B Fine-Tuning (Colab Only)

This project fine-tunes **meta-llama/Llama-3.2-3B** using LoRA and 4-bit quantization to predict job roles from resume summaries.  
The training pipeline is designed specifically for **Google Colab GPU environments** to enable memory-efficient fine-tuning of large language models.

## Overview

The model learns to classify a resume summary into its corresponding job role using a synthetic dataset hosted on Hugging Face.  
Fine-tuning is performed using **QLoRA** with the `trl` library’s `SFTTrainer`, allowing efficient training on limited GPU memory.

In [ ]:
!pip install -q --upgrade bitsandbytes==0.48.2 trl==0.25.1

In [ ]:

import os
import torch
from datetime import datetime

from datasets import load_dataset
from huggingface_hub import login
from google.colab import userdata

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

import wandb

In [ ]:
BASE_MODEL = "meta-llama/Llama-3.2-3B"
PROJECT_NAME = "resume-role"
HF_USER = "VictorHabila"

RUN_NAME = f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"
LOG_TO_WANDB = True
MAX_SEQUENCE_LENGTH = 512



In [ ]:

hf_token = userdata.get("HF_TOKEN")
login(hf_token, add_to_git_credential=True)


In [ ]:

wandb_api_key = userdata.get('WANDB_API_KEY')
os.environ["WANDB_API_KEY"] = wandb_api_key
wandb.login()


os.environ["WANDB_PROJECT"] = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "false"
os.environ["WANDB_WATCH"] = "false"

In [ ]:
if LOG_TO_WANDB:
  wandb.init(project=PROJECT_NAME, name=RUN_NAME)

In [ ]:
dataset = load_dataset(
    "VictorHabila/synthetic_resume_summaries",
    split="train"
)

print(len(dataset))
dataset[0]

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    use_fast=True
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"



tokenizer.chat_template = """{% for message in messages %}
{% if message['role'] == 'user' %}
<|start_header_id|>user<|end_header_id|>

{{ message['content'] }}<|eot_id|>
{% elif message['role'] == 'assistant' %}
<|start_header_id|>assistant<|end_header_id|>

{{ message['content'] }}<|eot_id|>
{% endif %}
{% endfor %}"""

In [ ]:
train = dataset.select(range(100))
val = dataset.select(range(100,150))
test = dataset.select(range(150,200))

In [ ]:
capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
)

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)

base_model.generation_config.pad_token_id = tokenizer.pad_token_id

print(
    f"Memory footprint: "
    f"{base_model.get_memory_footprint() / 1e6:.1f} MB"
)

In [ ]:
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.1

TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj"
]

lora_parameters = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

In [ ]:
train_parameters = SFTConfig(
    output_dir="resume_llama_output",

    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,

   
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    max_grad_norm=0.3,

 
    fp16=not use_bf16,
    bf16=use_bf16,

  
    max_length=MAX_SEQUENCE_LENGTH,

    
    logging_steps=10,
    save_strategy="epoch",

    report_to="wandb",
    run_name=RUN_NAME,
)

In [ ]:
def tokenize_dataset(example):
    tokens = tokenizer(
        example["job_summary"],
        truncation=True,
        max_length=MAX_SEQUENCE_LENGTH,
        padding="max_length"
    )
    return tokens

train = train.map(tokenize_dataset, batched=True)
val   = val.map(tokenize_dataset, batched=True)

In [ ]:
fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=train,
    eval_dataset=val,
    peft_config=lora_parameters,
    args=train_parameters

)

In [ ]:
fine_tuning.train()
fine_tuning.model.push_to_hub(PROJECT_RUN_NAME, private=True)
print(f"Saved to the hub: {PROJECT_RUN_NAME}")